# compare capabilities.json 
* determine which capabilities were added with which version of macOS / iOS 

In [1]:
# access different dir
import sys
sys.path.insert(1, '../capabilities-ipsw')

In [2]:
import json
import os

In [20]:
def load_json(file):
    with open(file) as f:
        return json.load(f)

# compare two json files
def diff_dict(file1, file2, path=""):
    diffs = []
    for key in file1.keys() | file2.keys():

        p = f"{path}.{key}" if path else key

        if key not in file1:
            diffs.append(f"Added {p}, {file2[key]}")
        elif key not in file2:
            diffs.append(f"Removed {p}, {file1[key]}")
        elif file1[key] != file2[key]:
            if isinstance(file1[key], dict) and isinstance(file2[key], dict):
                diffs.extend(diff_dict(file1[key], file2[key], p))
            else:
                diffs.append(f"Changed {p}: {file1[key]} → {file2[key]}")
    return diffs   

# enable comparison based on array with file names
def compare_capabilities(files):
    outputfile = f'{files[0].split('/')[2].strip('_Restore.ipsw')}-{files[1].split('/')[2].strip('_Restore.ipsw')}-comparison.txt'

    with open(outputfile, "w") as txtfile:
        for i in range(0, len(files)-1):
            file1 = files[i]
            file2 = files[i+1]
            txtfile.write(f"{file1} vs. {file2}\n")

            # parse json 
            file1_json = load_json(file1)
            file2_json = load_json(file2)
            
            for d in diff_dict(file1_json, file2_json):
                txtfile.write(f"{d}\n")


In [23]:
compare_capabilities(['../capabilities-ipsw/iPad_Pro_Spring_2021_18.4_22E240_Restore.ipsw/capabilities.json', '../capabilities-ipsw/iPhone14,4_18.4_22E240_Restore.ipsw/capabilities.json'])

In [5]:
# compare capabilities stored in dir
# the comparison sorts the files in the given directory and then compares each file with the subsequent one
# like this, also cross-device comparisons are made (which are of less value)
def compare_capability_files(path):
    outputfile = 'differences.txt'
    
    with open(outputfile, "w") as txtfile:
        file1 = ""
        file2 = ""

        # get list of all files 
        files = [f"{p}/{f[0]}" for p,d,f in sorted(os.walk(path))[1:]]

        for i in range(0, len(files)-1):
            file1 = files[i]
            file2 = files[i+1]
            txtfile.write(f"\n{file1} vs. {file2}\n")

            # parse json 
            file1_json = load_json(file1)
            file2_json = load_json(file2)
            
            for d in diff_dict(file1_json, file2_json):
                txtfile.write(f"{d}\n")


In [ ]:
# compares the capabilities and collects the changes per OS version 
# the list is still based on the comparison of subsequent ipsw capability extraction files, thus it might contain some distracting values
def list_changes_per_os_version(path, split_by_device=True):
    outputfile = 'changes-per-os-version.txt'
    differences = {}
    file1 = ""
    file2 = ""

    # get list of all files 
    files = [f"{p}/{f[0]}" for p,d,f in sorted(os.walk(path))[1:]]

    for i in range(0, len(files)-1):
        file1 = files[i]
        file2 = files[i+1]

        # determine OS version of the second file, used as key in the dict
        if split_by_device:
            os_version2 = f"{file2.split('/')[2].split('_')[0]}-{next(s for s in file2.split('/')[2].split('_') if "." in s)}"
        else:
            os_version2 = f"{file2.split('/')[2].split('_')[0].split('1')[0]}-{next(s for s in file2.split('/')[2].split('_') if "." in s)}"

        # parse json 
        file1_json = load_json(file1)
        file2_json = load_json(file2)

        diff = diff_dict(file1_json, file2_json)
        
        # add diff to dict if diff is not empty
        if diff != []:
            if differences.get(os_version2) == None:
                differences[os_version2] = [diff]
            else:
                differences[os_version2].append(diff)
        
    # write result to file in readable format
    with open(outputfile, "w") as txtfile:
        for key in differences.keys():
            txtfile.write(f"\n{key}:\n")
            for val in differences[key]:
                for capability in val:
                    txtfile.write(f"{capability},\n")
    return differences

    

In [51]:
compare_capability_files('../capabilities-ipsw')

In [50]:
diffs = list_changes_per_os_version('../capabilities-ipsw')